In [ ]:
!pip install -q transformers scikit-image networkx opencv-python-headless

In [ ]:
import os, cv2, heapq, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from scipy.spatial import KDTree
from skimage.morphology import skeletonize
from transformers import AutoModel
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Any
from google.colab import files
warnings.filterwarnings('ignore')
print(" All imports done")

 All imports done


In [ ]:
# ── STEP 1 : Upload your satellite image ──
print("Please upload your satellite image (e.g. 4208.png)")
uploaded_img = files.upload()
IMAGE_PATH = list(uploaded_img.keys())[0]
print(f" Image loaded: {IMAGE_PATH}")

Please upload your satellite image (e.g. 4208.png)


Saving 4208.png to 4208.png
 Image loaded: 4208.png


In [ ]:
# ── STEP 2 : Mount Google Drive (no need to upload large .pth file) ──
from google.colab import drive
drive.mount('/content/drive')

# ── Set the path to your checkpoint in Google Drive ──
WEIGHTS_PATH = "/content/drive/MyDrive/checkpoints/best_model.pth"

# Verify it exists
import os
if os.path.exists(WEIGHTS_PATH):
    print(f"Checkpoint found: {WEIGHTS_PATH}")
else:
    print("File not found — check your path above")

Mounted at /content/drive
Checkpoint found: /content/drive/MyDrive/checkpoints/best_model.pth


In [ ]:
# ── Config ──
IMAGE_SIZE  = 560
NUM_CLASSES = 8
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

LOVE_DA_COLORS = {
    0: (0,   0,   0),
    1: (255, 255, 255),
    2: (255, 0,   0),
    3: (255, 255, 0),
    4: (0,   0,   255),
    5: (139, 69,  19),
    6: (0,   255, 0),
    7: (0,   128, 0),
}

CLASS_NAMES = {
    0:"Ignore", 1:"Background", 2:"Building", 3:"Road",
    4:"Water",  5:"Barren",     6:"Forest",   7:"Agriculture"
}
print(f"Config ready | Device: {DEVICE}")

Config ready | Device: cuda


In [ ]:
class ResidualConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch))
        self.skip = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False),
                                   nn.BatchNorm2d(out_ch))
                     if in_ch != out_ch else nn.Identity())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.conv(x) + self.skip(x))


class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, rates=(6,12,18)):
        super().__init__()
        self.conv1    = nn.Sequential(nn.Conv2d(in_ch,out_ch,1,bias=False),
                                      nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True))
        self.dilated  = nn.ModuleList([
            nn.Sequential(nn.Conv2d(in_ch,out_ch,3,padding=r,dilation=r,bias=False),
                          nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True)) for r in rates])
        self.pool     = nn.Sequential(nn.AdaptiveAvgPool2d(1),
                                      nn.Conv2d(in_ch,out_ch,1,bias=False),
                                      nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True))
        n = len(rates)+2
        self.project  = nn.Sequential(nn.Conv2d(out_ch*n,out_ch,1,bias=False),
                                      nn.BatchNorm2d(out_ch),nn.ReLU(inplace=True),
                                      nn.Dropout2d(0.1))
    def forward(self, x):
        H,W = x.shape[-2:]
        out = [self.conv1(x)]
        for d in self.dilated: out.append(d(x))
        out.append(F.interpolate(self.pool(x),size=(H,W),mode="bilinear",align_corners=False))
        return self.project(torch.cat(out,dim=1))


class FPNFuse(nn.Module):
    def __init__(self, encoder_dim=768, fpn_dim=256):
        super().__init__()
        self.lat  = nn.ModuleList([
            nn.Sequential(nn.Conv2d(encoder_dim,fpn_dim,1,bias=False),
                          nn.BatchNorm2d(fpn_dim),nn.ReLU(inplace=True))
            for _ in range(4)])
        self.fuse = nn.Sequential(nn.Conv2d(fpn_dim*4,fpn_dim*2,1,bias=False),
                                  nn.BatchNorm2d(fpn_dim*2),nn.ReLU(inplace=True))
    def forward(self, feats):
        H,W = feats[0].shape[-2:]
        outs = []
        for i,f in enumerate(feats):
            x = self.lat[i](f)
            if x.shape[-2:] != (H,W):
                x = F.interpolate(x,size=(H,W),mode="bilinear",align_corners=False)
            outs.append(x)
        return self.fuse(torch.cat(outs,dim=1))


class StrongDINOv2Seg(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_size = 14
        self.encoder = AutoModel.from_pretrained("facebook/dinov2-base",
                                                  output_hidden_states=True)
        self.fpn  = FPNFuse(encoder_dim=768, fpn_dim=256)
        self.aspp = ASPP(in_ch=512, out_ch=256)
        self.up1  = nn.Upsample(scale_factor=2,mode="bilinear",align_corners=False)
        self.dec1 = ResidualConvBlock(256,256)
        self.up2  = nn.Upsample(scale_factor=2,mode="bilinear",align_corners=False)
        self.dec2 = ResidualConvBlock(256,128)
        self.up3  = nn.Upsample(scale_factor=2,mode="bilinear",align_corners=False)
        self.dec3 = ResidualConvBlock(128,64)
        self.up4  = nn.Upsample(scale_factor=2,mode="bilinear",align_corners=False)
        self.dec4 = ResidualConvBlock(64,32)
        self.seg_head = nn.Conv2d(32, NUM_CLASSES, 1)

    def _to_map(self, hidden, H, W):
        x = hidden[:,1:,:]
        B,N,C = x.shape
        return x.permute(0,2,1).reshape(B,C,H//self.patch_size,W//self.patch_size)

    def forward(self, x):
        B,_,H,W = x.shape
        enc    = self.encoder(pixel_values=x, output_hidden_states=True)
        layers = enc.hidden_states
        f3  = self._to_map(layers[3],  H, W)
        f6  = self._to_map(layers[6],  H, W)
        f9  = self._to_map(layers[9],  H, W)
        f12 = self._to_map(layers[12], H, W)
        neck = self.aspp(self.fpn([f3,f6,f9,f12]))
        d1 = self.dec1(self.up1(neck))
        d2 = self.dec2(self.up2(d1))
        d3 = self.dec3(self.up3(d2))
        d4 = self.dec4(self.up4(d3))
        d4 = F.interpolate(d4,size=(H,W),mode="bilinear",align_corners=False)
        return self.seg_head(d4)

print("Model classes defined")

Model classes defined


In [ ]:
 #── Load trained model ──
print("Loading model...")
model = StrongDINOv2Seg().to(DEVICE)
checkpoint = torch.load(WEIGHTS_PATH, map_location=DEVICE, weights_only=False)
print(f"Epoch : {checkpoint['epoch']}")
print(f"mIoU  : {checkpoint['mIoU']:.4f}")
print(f"mDice : {checkpoint['mDice']:.4f}")
missing, unexpected = model.load_state_dict(checkpoint["state_dict"], strict=False)
print("Missing Keys   :", missing)
print("Unexpected Keys:", unexpected)
model.eval()
print("Model ready")

Loading model...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Epoch : 4
mIoU  : 0.5380
mDice : 0.6891
Missing Keys   : []
Unexpected Keys: ['aux1.weight', 'aux1.bias', 'aux2.weight', 'aux2.bias', 'aux3.weight', 'aux3.bias']
Model ready


In [ ]:
# ── Segmentation Inference ──

def colorize_mask(mask):
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls, color in LOVE_DA_COLORS.items():
        rgb[mask == cls] = color
    return rgb

@torch.no_grad()
def predict_single(image_path):
    image     = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    resized   = cv2.resize(image_rgb, (IMAGE_SIZE, IMAGE_SIZE))

    pixel_values = (torch.tensor(resized.astype(np.float32)/255.0)
                    .permute(2,0,1).unsqueeze(0).to(DEVICE))

    logits = model(pixel_values)
    pred   = torch.argmax(logits, dim=1).squeeze().cpu().numpy()

    print("\nDetected Classes:")
    unique, counts = np.unique(pred, return_counts=True)
    for u, c in zip(unique, counts):
        #print(f"  {CLASS_NAMES[int(u)]:15s}: {c} px  ({100*c/pred.size:.1f}%)")
        DISPLAY_CLASS_NAMES = [

            'Background',
            'Building',
            'Road',
            'Water',
            'Barren',
            'Forest',
            'Agriculture',
            'Other'
        ]

        for u, c in zip(unique, counts):
            print(f"  {DISPLAY_CLASS_NAMES[int(u)]:15s}: {c} px  ({100*c/pred.size:.1f}%)")

    mask    = colorize_mask(pred)
    overlay = cv2.addWeighted(resized, 0.6, mask, 0.4, 0)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    axes[0].imshow(resized);  axes[0].set_title("Input");        axes[0].axis("off")
    axes[1].imshow(mask);     axes[1].set_title("Segmentation"); axes[1].axis("off")
    axes[2].imshow(overlay);  axes[2].set_title("Overlay");      axes[2].axis("off")
    plt.tight_layout()
    plt.savefig("segmentation_result.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Segmentation done")
    return resized, pred   # ← returns to planner

# Run prediction
original, pred = predict_single(IMAGE_PATH)


Detected Classes:
  Building       : 21604 px  (6.9%)
  Road           : 17745 px  (5.7%)
  Water          : 6161 px  (2.0%)
  Barren         : 34145 px  (10.9%)
  Agriculture    : 129573 px  (41.3%)
  Other          : 104372 px  (33.3%)
  Building       : 21604 px  (6.9%)
  Road           : 17745 px  (5.7%)
  Water          : 6161 px  (2.0%)
  Barren         : 34145 px  (10.9%)
  Agriculture    : 129573 px  (41.3%)
  Other          : 104372 px  (33.3%)
  Building       : 21604 px  (6.9%)
  Road           : 17745 px  (5.7%)
  Water          : 6161 px  (2.0%)
  Barren         : 34145 px  (10.9%)
  Agriculture    : 129573 px  (41.3%)
  Other          : 104372 px  (33.3%)
  Building       : 21604 px  (6.9%)
  Road           : 17745 px  (5.7%)
  Water          : 6161 px  (2.0%)
  Barren         : 34145 px  (10.9%)
  Agriculture    : 129573 px  (41.3%)
  Other          : 104372 px  (33.3%)
  Building       : 21604 px  (6.9%)
  Road           : 17745 px  (5.7%)
  Water          : 6161 px  (

In [ ]:
# ══════════════════════════════════════════════════════════
#  INFRASTRUCTURE PLANNER  — All 11 Stages
# ══════════════════════════════════════════════════════════

CLASS_BACKGROUND  = 1
CLASS_BUILDING    = 2
CLASS_ROAD        = 3
CLASS_WATER       = 4
CLASS_BARREN      = 5
CLASS_FOREST      = 6
CLASS_AGRICULTURE = 7

TRAVERSAL_COSTS = {
    CLASS_ROAD:        1,
    CLASS_AGRICULTURE: 2,
    CLASS_BACKGROUND:  3,
    CLASS_BARREN:      4,
    CLASS_FOREST:      8,
    CLASS_WATER:       np.inf,
    0:                 np.inf,
    CLASS_BUILDING:    5,
}

CLASS_COLORS_RGB = {
    0:(0,0,0),1:(128,128,128),2:(255,0,0),3:(255,255,0),
    4:(0,0,255),5:(165,42,42),6:(0,128,0),7:(144,238,144)
}

@dataclass
class Building:
    building_id: int
    centroid: Tuple[float,float]
    area: int
    bbox: Tuple[int,int,int,int]
    connected: bool = False
    nearest_road_pixel: Optional[Tuple[int,int]] = None
    nearest_road_dist: float = np.inf

@dataclass
class PlannerResults:
    building_list: List = field(default_factory=list)
    road_mask: Any = None
    road_skeleton: Any = None
    road_graph: Any = None
    connected_buildings: List = field(default_factory=list)
    isolated_buildings: List = field(default_factory=list)
    nearest_connections: Dict = field(default_factory=dict)
    cost_map: Any = None
    planned_paths: Dict = field(default_factory=dict)
    proposed_road_network: Any = None
    figures: Dict = field(default_factory=dict)
    metrics: Dict = field(default_factory=dict)

class InfrastructurePlanner:
    def __init__(self, original, pred,
                 min_building_area=20, road_kernel_size=3, connectivity_radius=5):
        self.original = original.copy()
        self.pred     = pred.copy().astype(np.int32)
        self.H, self.W = pred.shape
        self.min_building_area   = min_building_area
        self.road_kernel_size    = road_kernel_size
        self.connectivity_radius = connectivity_radius
        self.results = PlannerResults()

    def run(self):
        print("="*55)
        print("  Infrastructure Planner — Pipeline Start")
        print("="*55)
        self.results.building_list         = self._s1_buildings()
        self.results.road_mask             = self._s2_road()
        self.results.road_skeleton         = self._s3_skeleton()
        self.results.road_graph            = self._s4_graph()
        conn, iso                          = self._s5_connectivity()
        self.results.connected_buildings   = conn
        self.results.isolated_buildings    = iso
        self.results.nearest_connections   = self._s6_kdtree()
        self.results.cost_map              = self._s7_costmap()
        self.results.planned_paths         = self._s8_astar()
        self.results.proposed_road_network = self._s9_merge()
        self.results.figures               = self._s10_visualize()
        self.results.metrics               = self._s11_metrics()
        print("\n" + "="*55)
        print("  Pipeline Complete")
        print("="*55)
        return self.results

    # ── Stage 1 ─────────────────────────────────────────
    def _s1_buildings(self):
        print("\n[Stage 1] Building Detection")
        bm = (self.pred == CLASS_BUILDING).astype(np.uint8)
        n, labels, stats, centroids = cv2.connectedComponentsWithStats(bm, connectivity=8)
        bl = []
        bid = 1
        for lbl in range(1, n):
            area = int(stats[lbl, cv2.CC_STAT_AREA])
            if area < self.min_building_area: continue
            cx,cy = centroids[lbl]
            x=int(stats[lbl,cv2.CC_STAT_LEFT]); y=int(stats[lbl,cv2.CC_STAT_TOP])
            w=int(stats[lbl,cv2.CC_STAT_WIDTH]);h=int(stats[lbl,cv2.CC_STAT_HEIGHT])
            bl.append(Building(bid,(cy,cx),area,(y,x,y+h,x+w))); bid+=1
        print(f"  Detected {len(bl)} buildings")
        return bl

    # ── Stage 2 ─────────────────────────────────────────
    def _s2_road(self):
        print("[Stage 2] Road Extraction")
        rm = (self.pred == CLASS_ROAD).astype(np.uint8)
        k  = cv2.getStructuringElement(cv2.MORPH_RECT,(self.road_kernel_size,)*2)
        rm = cv2.morphologyEx(rm, cv2.MORPH_CLOSE, k)
        rm = cv2.morphologyEx(rm, cv2.MORPH_OPEN,  k)
        print(f"  Road pixels: {rm.sum():,}")
        return rm

    # ── Stage 3 ─────────────────────────────────────────
    def _s3_skeleton(self):
        print("[Stage 3] Skeletonization")
        sk = skeletonize(self.results.road_mask.astype(bool)).astype(np.uint8)
        print(f"  Skeleton pixels: {sk.sum():,}")
        return sk

    # ── Stage 4 ─────────────────────────────────────────
    def _s4_graph(self):
        print("[Stage 4] Road Graph Construction")
        sk = self.results.road_skeleton
        G  = nx.Graph()
        ps = set(zip(*np.where(sk>0)))
        G.add_nodes_from(ps)
        nb = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
        for r,c in ps:
            for dr,dc in nb:
                n=(r+dr,c+dc)
                if n in ps:
                    G.add_edge((r,c),n,weight=1.0 if dr==0 or dc==0 else 1.4142)
        print(f"  Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
        return G

    # ── Stage 5 ─────────────────────────────────────────
    def _s5_connectivity(self):
        print("[Stage 5] Connectivity Analysis")
        r   = self.connectivity_radius
        k   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*r+1,2*r+1))
        dil = cv2.dilate(self.results.road_mask, k, iterations=1)
        conn, iso = [], []
        for b in self.results.building_list:
            r0,c0,r1,c1 = b.bbox
            reg = dil[max(0,r0-r):min(self.H,r1+r), max(0,c0-r):min(self.W,c1+r)]
            if reg.any(): b.connected=True;  conn.append(b)
            else:         b.connected=False; iso.append(b)
        print(f"  Connected: {len(conn)}  |  Isolated: {len(iso)}")
        return conn, iso

    # ── Stage 6 ─────────────────────────────────────────
    def _s6_kdtree(self):
        print("[Stage 6] KDTree Nearest Road Search")
        sk  = self.results.road_skeleton
        iso = self.results.isolated_buildings
        rp  = np.column_stack(np.where(sk>0))
        if len(rp)==0: print("  WARNING: no road pixels"); return {}
        tree = KDTree(rp)
        nc   = {}
        for b in iso:
            dist,idx = tree.query(list(b.centroid))
            nc[b.building_id]     = tuple(rp[idx])
            b.nearest_road_pixel  = tuple(rp[idx])
            b.nearest_road_dist   = float(dist)
        print(f"  Mapped {len(nc)} isolated buildings to nearest road")
        return nc

    # ── Stage 7 ─────────────────────────────────────────
    def _s7_costmap(self):
        print("[Stage 7] Cost Map Generation")
        cm = np.full((self.H,self.W), np.inf)
        for cls,cost in TRAVERSAL_COSTS.items():
            cm[self.pred==cls] = cost
        print(f"  Traversable: {100*np.isfinite(cm).sum()/cm.size:.1f}%")
        return cm

    # ── Stage 8 ─────────────────────────────────────────
    def _s8_astar(self):
        print("[Stage 8] A* Road Planning")
        iso = self.results.isolated_buildings
        nc  = self.results.nearest_connections
        cm  = self.results.cost_map
        planned = {}
        for b in iso:
            start = (int(round(b.centroid[0])),int(round(b.centroid[1])))
            goal  = nc.get(b.building_id)
            if goal is None: continue
            goal  = (int(goal[0]),int(goal[1]))
            path  = self._astar(start,goal,cm)
            planned[b.building_id] = path if path else self._bresenham(start,goal)
        print(f"  Planned paths: {len(planned)}")
        return planned

    def _astar(self, start, goal, cm):
        H,W = cm.shape
        def h(a,b): return ((a[0]-b[0])**2+(a[1]-b[1])**2)**0.5
        heap = [(0.0,start)]; came={}; g={start:0.0}
        nb=[(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
        while heap:
            _,cur = heapq.heappop(heap)
            if cur==goal:
                path=[]
                while cur in came: path.append(cur); cur=came[cur]
                path.append(start); path.reverse(); return path
            cr,cc=cur
            for dr,dc in nb:
                nr,nc2=cr+dr,cc+dc
                if not(0<=nr<H and 0<=nc2<W): continue
                sc=cm[nr,nc2]
                if not np.isfinite(sc): continue
                md=1.0 if(dr==0 or dc==0) else 1.4142
                tg=g[cur]+sc*md; nb2=(nr,nc2)
                if tg<g.get(nb2,np.inf):
                    came[nb2]=cur; g[nb2]=tg
                    heapq.heappush(heap,(tg+h(nb2,goal),nb2))
        return None

    @staticmethod
    def _bresenham(start,end):
        r0,c0=start; r1,c1=end; path=[]
        dr=abs(r1-r0);dc=abs(c1-c0)
        sr=1 if r0<r1 else -1; sc=1 if c0<c1 else -1; err=dr-dc
        while True:
            path.append((r0,c0))
            if r0==r1 and c0==c1: break
            e2=2*err
            if e2>-dc: err-=dc; r0+=sr
            if e2<dr:  err+=dr; c0+=sc
        return path

    # ── Stage 9 ─────────────────────────────────────────
    def _s9_merge(self):
        print("[Stage 9] Merging Planned Roads")
        net = np.zeros((self.H,self.W),np.uint8)
        for path in self.results.planned_paths.values():
            for r,c in path:
                if 0<=r<self.H and 0<=c<self.W: net[r,c]=1
        net = cv2.dilate(net,np.ones((2,2),np.uint8),iterations=1)
        print(f"  Network pixels: {net.sum():,}")
        return net

    # ── Stage 10 ─────────────────────────────────────────
    def _s10_visualize(self):
        print("[Stage 10] Generating Visualizations")
        orig=self.original; pred=self.pred
        conn=self.results.connected_buildings
        iso =self.results.isolated_buildings
        rm  =self.results.road_mask
        sk  =self.results.road_skeleton
        pp  =self.results.planned_paths
        net =self.results.proposed_road_network
        figs={}

        def seg_color():
            img=np.zeros((*pred.shape,3),dtype=np.uint8)
            for cls,rgb in CLASS_COLORS_RGB.items(): img[pred==cls]=rgb
            return img

        # Fig 1 — Building Detection
        fig,ax=plt.subplots(figsize=(8,8))
        ax.imshow(orig)
        for b in conn:
            r0,c0,r1,c1=b.bbox
            ax.add_patch(mpatches.Rectangle((c0,r0),c1-c0,r1-r0,lw=1.2,ec="lime",fc="none"))
        for b in iso:
            r0,c0,r1,c1=b.bbox
            ax.add_patch(mpatches.Rectangle((c0,r0),c1-c0,r1-r0,lw=1.2,ec="red",fc="none"))
        ax.set_title("Stage 1 — Building Detection",fontsize=11); ax.axis("off")
        ax.legend(handles=[mpatches.Patch(color="lime",label="Connected"),
                            mpatches.Patch(color="red", label="Isolated")],
                  loc="upper right",fontsize=8)
        plt.tight_layout(); plt.savefig("fig1_buildings.png",dpi=120,bbox_inches="tight")
        plt.show(); figs["Building Detection"]=fig

        # Fig 2 — Skeleton
        fig,axes=plt.subplots(1,2,figsize=(14,6))
        axes[0].imshow(rm,cmap="gray"); axes[0].set_title("Road Mask"); axes[0].axis("off")
        axes[1].imshow(orig)
        ov=np.zeros((*sk.shape,4),dtype=np.float32); ov[sk>0]=[1,1,0,0.85]
        axes[1].imshow(ov); axes[1].set_title("Stage 3 — Skeleton"); axes[1].axis("off")
        plt.tight_layout(); plt.savefig("fig2_skeleton.png",dpi=120,bbox_inches="tight")
        plt.show(); figs["Road Skeleton"]=fig

        # Fig 3 — Connectivity
        fig,ax=plt.subplots(figsize=(8,8))
        ax.imshow(seg_color(),alpha=0.6); ax.imshow(orig,alpha=0.4)
        for b in conn: ax.plot(b.centroid[1],b.centroid[0],"g^",ms=5)
        for b in iso:
            ax.plot(b.centroid[1],b.centroid[0],"rv",ms=5)
            if b.nearest_road_pixel:
                rp,cp=b.nearest_road_pixel
                ax.plot([b.centroid[1],cp],[b.centroid[0],rp],"w--",lw=0.7,alpha=0.7)
        ax.set_title("Stage 5&6 — Connectivity + KDTree Links",fontsize=11); ax.axis("off")
        ax.legend(handles=[mpatches.Patch(color="green",label=f"Connected ({len(conn)})"),
                            mpatches.Patch(color="red",  label=f"Isolated ({len(iso)})")],
                  loc="upper right",fontsize=8)
        plt.tight_layout(); plt.savefig("fig3_connectivity.png",dpi=120,bbox_inches="tight")
        plt.show(); figs["Connectivity Map"]=fig

        # Fig 4 — A* Paths
        fig,ax=plt.subplots(figsize=(8,8))
        ax.imshow(orig,alpha=0.7)
        po=np.zeros((*pred.shape,4),dtype=np.float32)
        for path in pp.values():
            for r,c in path:
                if 0<=r<self.H and 0<=c<self.W: po[r,c]=[1,0.2,0,0.9]
        ax.imshow(po); ax.set_title("Stage 8 — A* Planned Paths",fontsize=11); ax.axis("off")
        plt.tight_layout(); plt.savefig("fig4_astar.png",dpi=120,bbox_inches="tight")
        plt.show(); figs["A* Paths"]=fig

        # Fig 5 — Final
        fig,ax=plt.subplots(figsize=(10,10))
        ax.imshow(orig,alpha=0.65)
        ro=np.zeros((*pred.shape,4),dtype=np.float32); ro[rm>0]=[1,1,0,0.5];  ax.imshow(ro)
        no=np.zeros((*pred.shape,4),dtype=np.float32); no[net>0]=[1,0,1,0.85]; ax.imshow(no)
        for b in conn:
            r0,c0,r1,c1=b.bbox
            ax.add_patch(mpatches.Rectangle((c0,r0),c1-c0,r1-r0,lw=1,ec="lime",fc="none"))
        for b in iso:
            r0,c0,r1,c1=b.bbox
            ax.add_patch(mpatches.Rectangle((c0,r0),c1-c0,r1-r0,lw=1,ec="red",fc="none"))
        ax.set_title("Stage 10 — Final Infrastructure Expansion",fontsize=11); ax.axis("off")
        ax.legend(handles=[mpatches.Patch(color="yellow", label="Existing Roads"),
                            mpatches.Patch(color="magenta",label="Proposed Roads"),
                            mpatches.Patch(color="lime",   label="Connected Buildings"),
                            mpatches.Patch(color="red",    label="Isolated Buildings")],
                  loc="upper right",fontsize=8)
        plt.tight_layout(); plt.savefig("fig5_final.png",dpi=120,bbox_inches="tight")
        plt.show(); figs["Final Expansion"]=fig
        return figs

    # ── Stage 11 ─────────────────────────────────────────
    def _s11_metrics(self):
        print("[Stage 11] Metrics")

        bl   = self.results.building_list
        conn = self.results.connected_buildings
        iso  = self.results.isolated_buildings
        pp   = self.results.planned_paths

        total = len(bl)
        nc    = len(conn)
        ni    = len(iso)

        # --------------------------------------------------
        # 1. Connectivity before planning
        # --------------------------------------------------
        connectivity_before = (
            nc / total * 100
            if total > 0 else 0
        )

        # --------------------------------------------------
        # 2. Connectivity after planning
        # --------------------------------------------------
        newly_connected = len(pp)

        connectivity_after = (
            (nc + newly_connected) / total * 100
            if total > 0 else 0
        )

        # --------------------------------------------------
        # 3. Nearest-road connection distances
        # --------------------------------------------------
        connection_distances = [
            b.nearest_road_dist
            for b in iso
            if np.isfinite(b.nearest_road_dist)
        ]

        avg_connection_distance = (
            float(np.mean(connection_distances))
            if connection_distances else 0.0
        )

        # --------------------------------------------------
        # 4. Calculate actual A* path lengths
        # --------------------------------------------------
        path_lengths = []

        for building_id, path in pp.items():

            if path is None or len(path) < 2:
                continue

            length = 0.0

            for i in range(len(path) - 1):

                r1, c1 = path[i]
                r2, c2 = path[i + 1]

                step = np.sqrt(
                    (r2 - r1) ** 2 +
                    (c2 - c1) ** 2
                )

                length += step

            path_lengths.append(length)

        # --------------------------------------------------
        # 5. Total and average proposed road length
        # --------------------------------------------------
        total_path_length = (
            float(np.sum(path_lengths))
            if path_lengths else 0.0
        )

        average_path_length = (
            float(np.mean(path_lengths))
            if path_lengths else 0.0
        )

        # --------------------------------------------------
        # 6. Path efficiency
        #
        # Euclidean distance from building to nearest road
        # divided by actual A* path length.
        # --------------------------------------------------
        efficiencies = []

        for b in iso:

            if b.building_id not in pp:
                continue

            path = pp[b.building_id]

            if path is None or len(path) < 2:
                continue

            actual_length = 0.0

            for i in range(len(path) - 1):

                r1, c1 = path[i]
                r2, c2 = path[i + 1]

                actual_length += np.sqrt(
                    (r2 - r1) ** 2 +
                    (c2 - c1) ** 2
                )

            euclidean_distance = float(b.nearest_road_dist)

            if actual_length > 0 and np.isfinite(euclidean_distance):

                efficiency = euclidean_distance / actual_length

                efficiencies.append(efficiency)

        average_path_efficiency = (
            float(np.mean(efficiencies))
            if efficiencies else 0.0
        )

        # --------------------------------------------------
        # 7. Metrics dictionary
        # --------------------------------------------------
        m = {

            "total_buildings":
                total,

            "connected_buildings_before":
                nc,

            "isolated_buildings_before":
                ni,

            "connectivity_before_pct":
                round(connectivity_before, 2),

            "newly_connected_buildings":
                newly_connected,

            "connected_buildings_after":
                min(nc + newly_connected, total),

            "connectivity_after_pct":
                round(min(connectivity_after, 100), 2),

            "proposed_new_road_connections":
                len(pp),

            "total_proposed_road_length_px":
                round(total_path_length, 2),

            "average_proposed_road_length_px":
                round(average_path_length, 2),

            "avg_connection_distance_px":
                round(avg_connection_distance, 2),

            "average_path_efficiency":
                round(average_path_efficiency, 4),
        }

        # --------------------------------------------------
        # 8. Print report
        # --------------------------------------------------
        print("\n" + "─" * 55)
        print("  INFRASTRUCTURE EXPANSION REPORT")
        print("─" * 55)

        for key, value in m.items():

            label = key.replace("_", " ").title()

            if isinstance(value, float):
                print(f"  {label:<42} {value:.2f}")
            else:
                print(f"  {label:<42} {value}")

        print("─" * 55)

        return m
    # def _s11_metrics(self):
     #   print("[Stage 11] Metrics")
      #  bl=self.results.building_list
       # conn=self.results.connected_buildings
        # iso=self.results.isolated_buildings
        # pp=self.results.planned_paths
        # net=self.results.proposed_road_network
        # total=len(bl); nc=len(conn); ni=len(iso)
        #cb = nc/total*100 if total else 0
        #ca = (nc+len(pp))/total*100 if total else 0
        #dists=[b.nearest_road_dist for b in iso if np.isfinite(b.nearest_road_dist)]
        #m = {
         #   "total_buildings":              total,
          #  "connected_buildings":          nc,
           # "disconnected_buildings":       ni,
            #"connectivity_before_pct":      round(cb,2),
            #"connectivity_after_pct":       round(ca,2),
            #"proposed_new_road_connections":len(pp),
            #"total_proposed_road_length_px":int(net.sum()),
            #"avg_connection_distance_px":   round(float(np.mean(dists)) if dists else 0,2),
        #}
        #print("\n" + "─"*50)
        #print("  INFRASTRUCTURE EXPANSION REPORT")
        #print("─"*50)
        #for k,v in m.items():
         #   label=k.replace("_"," ").title()
          #  print(f"  {label:<40} {v:.2f}" if isinstance(v,float) else f"  {label:<40} {v}")
        #print("─"*50)
        #return m


print("InfrastructurePlanner class defined")

InfrastructurePlanner class defined


In [ ]:
# ── Run Infrastructure Planner ──


#planner = InfrastructurePlanner(
 #   original,
  #  pred,
   # min_building_area=20,
    #road_kernel_size=3,
    #connectivity_radius=5
#)
#results = planner.run()"""

# Run prediction
original, pred = predict_single(IMAGE_PATH)

# ── Run Infrastructure Planner ──
CONNECTIVITY_RADIUS = 5

planner = InfrastructurePlanner(
    original,
    pred,
    min_building_area=20,
    road_kernel_size=3,
    connectivity_radius=CONNECTIVITY_RADIUS
)

results = planner.run()

print("\n========== PLANNING RESULTS ==========")
print(results)


Detected Classes:
  Building       : 21604 px  (6.9%)
  Road           : 17745 px  (5.7%)
  Water          : 6161 px  (2.0%)
  Barren         : 34145 px  (10.9%)
  Agriculture    : 129573 px  (41.3%)
  Other          : 104372 px  (33.3%)
  Building       : 21604 px  (6.9%)
  Road           : 17745 px  (5.7%)
  Water          : 6161 px  (2.0%)
  Barren         : 34145 px  (10.9%)
  Agriculture    : 129573 px  (41.3%)
  Other          : 104372 px  (33.3%)
  Building       : 21604 px  (6.9%)
  Road           : 17745 px  (5.7%)
  Water          : 6161 px  (2.0%)
  Barren         : 34145 px  (10.9%)
  Agriculture    : 129573 px  (41.3%)
  Other          : 104372 px  (33.3%)
  Building       : 21604 px  (6.9%)
  Road           : 17745 px  (5.7%)
  Water          : 6161 px  (2.0%)
  Barren         : 34145 px  (10.9%)
  Agriculture    : 129573 px  (41.3%)
  Other          : 104372 px  (33.3%)
  Building       : 21604 px  (6.9%)
  Road           : 17745 px  (5.7%)
  Water          : 6161 px  (

In [ ]:
print("\n========== PLANNING METRICS ==========")

for key, value in results.metrics.items():
    print(f"{key}: {value}")


========== PLANNING METRICS ==========
total_buildings: 15
connected_buildings_before: 7
isolated_buildings_before: 8
connectivity_before_pct: 46.67
newly_connected_buildings: 8
connected_buildings_after: 15
connectivity_after_pct: 100.0
proposed_new_road_connections: 8
total_proposed_road_length_px: 1130.1
average_proposed_road_length_px: 141.26
avg_connection_distance_px: 111.98
average_path_efficiency: 0.8502


In [ ]:
# ============================================================
# CONNECTIVITY RADIUS SENSITIVITY ANALYSIS
# ============================================================

radius_values = [3, 5, 8, 10]
radius_results = {}

print("\n" + "="*65)
print("       CONNECTIVITY RADIUS SENSITIVITY ANALYSIS")
print("="*65)

for radius in radius_values:

    print(f"\n\n>>> Testing connectivity radius = {radius}")

    planner_test = InfrastructurePlanner(
        original,
        pred,
        min_building_area=20,
        road_kernel_size=3,
        connectivity_radius=radius
    )

    result_test = planner_test.run()

    radius_results[radius] = result_test.metrics

    print(f"\nRadius {radius} Results:")
    print(f"  Buildings                : {result_test.metrics['total_buildings']}")
    print(f"  Connected before        : {result_test.metrics['connected_buildings_before']}")
    print(f"  Connectivity before (%) : {result_test.metrics['connectivity_before_pct']}")
    print(f"  Isolated before         : {result_test.metrics['isolated_buildings_before']}")
    print(f"  Proposed connections    : {result_test.metrics['proposed_new_road_connections']}")
    print(f"  Connectivity after (%)  : {result_test.metrics['connectivity_after_pct']}")
    print(f"  Total road length (px)  : {result_test.metrics['total_proposed_road_length_px']}")
    print(f"  Avg road length (px)    : {result_test.metrics['average_proposed_road_length_px']}")
    print(f"  Avg connection distance : {result_test.metrics['avg_connection_distance_px']}")
    print(f"  Path efficiency         : {result_test.metrics['average_path_efficiency']}")

print("\n" + "="*65)
print("             FINAL SENSITIVITY SUMMARY")
print("="*65)

print(
    f"{'Radius':<10}"
    f"{'Before %':<12}"
    f"{'After %':<12}"
    f"{'Proposed':<12}"
    f"{'Road Length':<15}"
    f"{'Efficiency':<12}"
)

print("-"*65)

for radius in radius_values:

    m = radius_results[radius]

    print(
        f"{radius:<10}"
        f"{m['connectivity_before_pct']:<12}"
        f"{m['connectivity_after_pct']:<12}"
        f"{m['proposed_new_road_connections']:<12}"
        f"{m['total_proposed_road_length_px']:<15}"
        f"{m['average_path_efficiency']:<12}"
    )


       CONNECTIVITY RADIUS SENSITIVITY ANALYSIS


>>> Testing connectivity radius = 3
  Infrastructure Planner — Pipeline Start

[Stage 1] Building Detection
  Detected 15 buildings
[Stage 2] Road Extraction
  Road pixels: 6,158
[Stage 3] Skeletonization
  Skeleton pixels: 564
[Stage 4] Road Graph Construction
  Graph: 564 nodes, 559 edges
[Stage 5] Connectivity Analysis
  Connected: 6  |  Isolated: 9
[Stage 6] KDTree Nearest Road Search
  Mapped 9 isolated buildings to nearest road
[Stage 7] Cost Map Generation
  Traversable: 89.1%
[Stage 8] A* Road Planning
  Planned paths: 9
[Stage 9] Merging Planned Roads
  Network pixels: 2,151
[Stage 10] Generating Visualizations
[Stage 11] Metrics

───────────────────────────────────────────────────────
  INFRASTRUCTURE EXPANSION REPORT
───────────────────────────────────────────────────────
  Total Buildings                            15
  Connected Buildings Before                 6
  Isolated Buildings Before                  9
  Connectivit

In [ ]:
# ============================================================
# PATH LENGTH CALCULATION
# ============================================================

def calculate_path_length(path):
    """
    Calculate the length of an A* path in pixels.
    Diagonal movements are measured using Euclidean distance.
    """

    if path is None or len(path) < 2:
        return 0.0

    total_length = 0.0

    for i in range(len(path) - 1):

        y1, x1 = path[i]
        y2, x2 = path[i + 1]

        step_length = np.sqrt(
            (x2 - x1) ** 2 +
            (y2 - y1) ** 2
        )

        total_length += step_length

    return float(total_length)


print("Path-length function ready.")

Path-length function ready.


In [ ]:
# ── Download all output figures ──
output_files = [
    "segmentation_result.png",
    "fig1_buildings.png",
    "fig2_skeleton.png",
    "fig3_connectivity.png",
    "fig4_astar.png",
    "fig5_final.png",
]
for f in output_files:
    if os.path.exists(f):
        files.download(f)
        print(f"Downloaded: {f}")
    else:
        print(f"Not found: {f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: segmentation_result.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: fig1_buildings.png


... (8 outputs left out)

Downloaded: fig4_astar.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded: fig5_final.png


In [ ]:
import os

# Check all expected output files
files_to_check = [
    "segmentation_result.png",
    "fig1_buildings.png",
    "fig2_skeleton.png",
    "fig3_connectivity.png",
    "fig4_astar.png",
    "fig5_final.png",
]

for f in files_to_check:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) if exists else 0
    status = f" {size/1024:.1f} KB" if exists else " NOT FOUND"
    print(f"{f:<35} {status}")

segmentation_result.png              2539.5 KB
fig1_buildings.png                   1367.5 KB
fig2_skeleton.png                    885.4 KB
fig3_connectivity.png                1059.8 KB
fig4_astar.png                       1234.6 KB
fig5_final.png                       1717.3 KB


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  STAGE 12 — FINAL METRICS DASHBOARD (White Theme)
#  Requires: results (from planner.run())  +  pred
# ══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

# ── Pull metrics ──────────────────────────────────────────────────────
#m = results.metrics

#total_buildings     = m.get("total_buildings", 0)
#connected_buildings = m.get("connected_buildings", 0)

# Correct isolated building count
#isolated_buildings = max(
  #  0,
 #   total_buildings - connected_buildings
#)

# Correct connectivity before planning
#connectivity_before = (
  #  100 * connected_buildings / total_buildings
 #   if total_buildings > 0 else 0.0
#)
#connectivity_after  = m.get("connectivity_after_pct", 0.0)
#proposed_road_px    = m.get("proposed_road_pixels", 0)
#proposed_road_len   = m.get("proposed_road_length_m", proposed_road_px)
#avg_conn_dist       = m.get("avg_connection_distance", 0.0)
#planned_paths_count = len(results.planned_paths) if results.planned_paths else 0
#improvement         = connectivity_after - connectivity_before

# ── Pull metrics ──────────────────────────────────────────────────────
m = results.metrics

total_buildings = m.get("total_buildings", 0)

# Metrics generated by Stage 11
connected_buildings = m.get("connected_buildings_before", 0)

isolated_buildings = m.get(
    "isolated_buildings_before",
    max(0, total_buildings - connected_buildings)
)

connectivity_before = m.get(
    "connectivity_before_pct",
    100 * connected_buildings / total_buildings
    if total_buildings > 0 else 0.0
)

connectivity_after = m.get("connectivity_after_pct", 0.0)

proposed_road_len = m.get(
    "total_proposed_road_length_px", 0.0
)

avg_conn_dist = m.get(
    "avg_connection_distance_px", 0.0
)

planned_paths_count = m.get(
    "proposed_new_road_connections",
    len(results.planned_paths) if results.planned_paths else 0
)

improvement = connectivity_after - connectivity_before

# ── Land cover from pred ──────────────────────────────────────────────
CLASS_NAMES_SHORT = {
    0:"Ignore", 1:"Background", 2:"Building", 3:"Road",
    4:"Water",  5:"Barren",     6:"Forest",   7:"Agriculture"
}
CLASS_COLORS = {
    0:"#555555", 1:"#aaaaaa", 2:"#e63946", 3:"#f4a300",
    4:"#1a73e8", 5:"#8b4513", 6:"#2ecc71", 7:"#27ae60"
}

unique, counts = np.unique(pred, return_counts=True)
total_px   = pred.size
land_names  = [CLASS_NAMES_SHORT[int(u)] for u in unique]
land_counts = counts
land_pct    = [100 * c / total_px for c in counts]
land_colors = [CLASS_COLORS[int(u)] for u in unique]

# ── Theme colors (all for white background) ───────────────────────────
BG     = "white"
PANEL  = "#f7f7f7"
BORDER = "#cccccc"
TEXT   = "#1a1a1a"
ACCENT = "#1a73e8"
GREEN  = "#1a7f3c"
RED    = "#c62828"
GOLD   = "#c77700"
LGRID  = "#e0e0e0"

# ── Figure setup ──────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 22), facecolor=BG)
fig.suptitle(
    "DINOv2 Rural Infrastructure Planner — Final Metrics Dashboard",
    fontsize=18, fontweight="bold", color=TEXT, y=0.98
)

gs = gridspec.GridSpec(4, 3, figure=fig,
                       hspace=0.50, wspace=0.35,
                       top=0.94, bottom=0.04)

def panel_ax(row, col, colspan=1):
    ax = fig.add_subplot(gs[row, col:col+colspan])
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_color(BORDER)
        sp.set_linewidth(1.0)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.title.set_color(TEXT)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    return ax

# ══ ROW 0 — KPI Cards ════════════════════════════════════════════════
kpis = [
    ("Total\nBuildings",   str(total_buildings),    ACCENT),
    ("Connected",          str(connected_buildings), GREEN),
    ("Isolated\n(Before)", str(isolated_buildings),  RED),
]
for col, (label, value, color) in enumerate(kpis):
    ax = panel_ax(0, col)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.62, value, ha="center", va="center",
            fontsize=44, fontweight="bold", color=color,
            transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha="center", va="center",
            fontsize=13, color=TEXT, transform=ax.transAxes)
    for sp in ax.spines.values():
        sp.set_linewidth(2.5)
        sp.set_color(color)

# ══ ROW 1 — Connectivity | Donut | Summary ═══════════════════════════

# 1A: Connectivity bar
ax1a = panel_ax(1, 0)
bars = ax1a.bar(
    ["Before\nPlanning", "After\nPlanning"],
    [connectivity_before, connectivity_after],
    color=[RED, GREEN], width=0.5, edgecolor=BORDER, linewidth=1.0
)
for bar, val in zip(bars, [connectivity_before, connectivity_after]):
    ax1a.text(bar.get_x() + bar.get_width()/2,
              bar.get_height() + 1.2,
              f"{val:.1f}%",
              ha="center", fontsize=13, fontweight="bold", color=TEXT)
ax1a.set_ylim(0, max(connectivity_after * 1.25, 10))
ax1a.set_title("Connectivity (%)", fontsize=11, fontweight="bold")
ax1a.set_ylabel("Buildings Connected (%)", fontsize=9)
ax1a.axhline(100, color=LGRID, lw=0.8, ls="--")
ax1a.yaxis.grid(True, color=LGRID, linewidth=0.5)
ax1a.set_axisbelow(True)
if improvement > 0:
    ax1a.annotate(
        f"+{improvement:.1f}%",
        xy=(1, connectivity_after),
        xytext=(0.55, (connectivity_before + connectivity_after) / 2),
        textcoords="data",
        fontsize=11, color=GOLD, fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=GOLD, lw=1.5)
    )

# 1B: Land cover donut
ax1b = panel_ax(1, 1)
ax1b.set_facecolor(BG)
ax1b.set_aspect("equal")
wedge_props = dict(width=0.45, edgecolor="white", linewidth=1.5)
ax1b.pie(land_counts, colors=land_colors,
         wedgeprops=wedge_props, startangle=90)
ax1b.text(0, 0, f"{total_px//1000}K\npx",
          ha="center", va="center", fontsize=10,
          color=TEXT, fontweight="bold")
ax1b.set_title("Land Cover Distribution", fontsize=11, fontweight="bold")
legend_patches = [
    mpatches.Patch(color=c, label=f"{n} ({p:.1f}%)")
    for c, n, p in zip(land_colors, land_names, land_pct)
]
ax1b.legend(handles=legend_patches, loc="lower center",
            bbox_to_anchor=(0.5, -0.30), ncol=2,
            fontsize=7.5, framealpha=0, labelcolor=TEXT)

# 1C: Road planning summary
ax1c = panel_ax(1, 2)
ax1c.set_xlim(0, 1); ax1c.set_ylim(0, 1)
ax1c.set_xticks([]); ax1c.set_yticks([])
stats_lines = [
    ("Planned Paths",           f"{planned_paths_count}",       ACCENT),
    ("Proposed Road Length",    f"{int(proposed_road_len)} px",  GOLD),
    ("Avg Connection Distance", f"{avg_conn_dist:.1f} px",       GREEN),
    ("Connectivity Gain",       f"+{improvement:.1f}%",          GOLD),
]
for i, (label, val, color) in enumerate(stats_lines):
    y = 0.82 - i * 0.20
    ax1c.text(0.05, y, label, color=TEXT,  fontsize=10, va="center")
    ax1c.text(0.95, y, val,   color=color, fontsize=12, va="center",
              ha="right", fontweight="bold")
    ax1c.axhline(y - 0.07, color=LGRID, lw=0.8, xmin=0.02, xmax=0.98)
ax1c.set_title("Road Planning Summary", fontsize=11, fontweight="bold")

# ══ ROW 2 — Land Bars | Histogram | Gauge ════════════════════════════

# 2A: Horizontal land cover
ax2a = panel_ax(2, 0)
y_pos = np.arange(len(land_names))
hbars = ax2a.barh(y_pos, land_pct, color=land_colors,
                  edgecolor=BORDER, linewidth=0.7, height=0.6)
ax2a.set_yticks(y_pos)
ax2a.set_yticklabels(land_names, fontsize=9)
ax2a.set_xlabel("Coverage (%)", fontsize=9)
ax2a.set_title("Land Cover by Class", fontsize=11, fontweight="bold")
ax2a.xaxis.grid(True, color=LGRID, linewidth=0.5)
ax2a.set_axisbelow(True)
for bar, pct in zip(hbars, land_pct):
    ax2a.text(bar.get_width() + 0.3,
              bar.get_y() + bar.get_height() / 2,
              f"{pct:.1f}%", va="center", fontsize=8, color=TEXT)

# 2B: Building area histogram
ax2b = panel_ax(2, 1)
if results.building_list:
    areas      = [b.area for b in results.building_list]
    conn_areas = [b.area for b in results.building_list if b.connected]
    iso_areas  = [b.area for b in results.building_list if not b.connected]
    bins = np.linspace(min(areas), max(areas) + 1, 20)
    if conn_areas:
        ax2b.hist(conn_areas, bins=bins, color=GREEN, alpha=0.75,
                  label=f"Connected ({len(conn_areas)})", edgecolor="white")
    if iso_areas:
        ax2b.hist(iso_areas, bins=bins, color=RED, alpha=0.75,
                  label=f"Isolated ({len(iso_areas)})", edgecolor="white")
    ax2b.set_xlabel("Building Area (px)", fontsize=9)
    ax2b.set_ylabel("Count", fontsize=9)
    ax2b.legend(fontsize=8, framealpha=1, facecolor=PANEL,
                edgecolor=BORDER, labelcolor=TEXT)
    ax2b.yaxis.grid(True, color=LGRID, linewidth=0.5)
    ax2b.set_axisbelow(True)
else:
    ax2b.text(0.5, 0.5, "No Buildings\nDetected",
              ha="center", va="center", color=TEXT, fontsize=13)
ax2b.set_title("Building Size Distribution", fontsize=11, fontweight="bold")

# 2C: Connectivity gauge
ax2c = panel_ax(2, 2)
ax2c.set_facecolor(BG)
ax2c.set_xlim(-1.3, 1.3); ax2c.set_ylim(-0.2, 1.3)
ax2c.set_xticks([]); ax2c.set_yticks([])
ax2c.set_aspect("equal")

theta_bg  = np.linspace(np.pi, 0, 200)
theta_bef = np.linspace(np.pi, np.pi - np.pi * connectivity_before / 100, 200)
theta_aft = np.linspace(np.pi, np.pi - np.pi * connectivity_after  / 100, 200)

ax2c.plot(np.cos(theta_bg),  np.sin(theta_bg),  color=LGRID,  lw=18, solid_capstyle="round")
ax2c.plot(np.cos(theta_bef), np.sin(theta_bef), color=RED,    lw=18, solid_capstyle="round", alpha=0.5)
ax2c.plot(np.cos(theta_aft), np.sin(theta_aft), color=GREEN,  lw=18, solid_capstyle="round")

ax2c.text(0, 0.30, f"{connectivity_after:.1f}%",
          ha="center", va="center", fontsize=26,
          fontweight="bold", color=GREEN)
ax2c.text(0, 0.05, "After Planning",
          ha="center", va="center", fontsize=10, color=TEXT)
ax2c.text(-1.1, -0.10, "0%",   ha="center", fontsize=9, color=TEXT)
ax2c.text( 1.1, -0.10, "100%", ha="center", fontsize=9, color=TEXT)
ax2c.set_title("Connectivity Gauge", fontsize=11, fontweight="bold")

# ══ ROW 3 — Summary Table ════════════════════════════════════════════
ax3 = panel_ax(3, 0, colspan=3)
ax3.set_xlim(0, 1); ax3.set_ylim(0, 1)
ax3.set_xticks([]); ax3.set_yticks([])
ax3.set_title("Infrastructure Planning Report — Summary",
              fontsize=12, fontweight="bold")

headers = ["Metric", "Value", "Status"]
rows = [
    ("Total Buildings Detected",     str(total_buildings),                          "—"),
    ("Connected Buildings", f"{connected_buildings}  ({connectivity_before:.1f}%)",
     "✓" if connectivity_before >= 50 else "⚠"),
    ("Isolated Buildings (Before)",  str(isolated_buildings),
     "⚠" if isolated_buildings > 0 else "✓"),
    ("Connectivity After Planning",  f"{connectivity_after:.1f}%",
     "✓" if connectivity_after >= 80 else "⚠"),
    ("Connectivity Improvement",     f"+{improvement:.1f}%",                        "✓"),
    ("Planned Road Paths",           str(planned_paths_count),
     "✓" if planned_paths_count > 0 else "—"),
    ("Proposed Road Length",         f"{int(proposed_road_len)} pixels",            "✓"),
    ("Avg Connection Distance",      f"{avg_conn_dist:.1f} px",                     "✓"),
]

col_x = [0.03, 0.52, 0.88]
row_h = 1.0 / (len(rows) + 2)

for cx, h in zip(col_x, headers):
    ax3.text(cx, 1 - row_h * 0.6, h,
             color=ACCENT, fontsize=10, fontweight="bold", va="center")
ax3.axhline(1 - row_h, color=BORDER, lw=1.2, xmin=0.01, xmax=0.99)

status_colors = {"✓": GREEN, "⚠": GOLD, "—": TEXT}
for i, (label, val, status) in enumerate(rows):
    y = 1 - row_h * (i + 1.8)
    bg_color = PANEL if i % 2 == 0 else "white"
    ax3.axhspan(y - row_h * 0.45, y + row_h * 0.55,
                color=bg_color, zorder=0)
    ax3.text(col_x[0], y, label,  color=TEXT,  fontsize=9,  va="center")
    ax3.text(col_x[1], y, val,    color=TEXT,  fontsize=9,  va="center")
    ax3.text(col_x[2], y, status,
             color=status_colors.get(status, TEXT),
             fontsize=11, va="center", fontweight="bold")
    if i < len(rows) - 1:
        ax3.axhline(y - row_h * 0.45, color=LGRID,
                    lw=0.6, xmin=0.01, xmax=0.99)

# ── Save & download ───────────────────────────────────────────────────
plt.savefig("fig12_metrics_dashboard.png", dpi=150,
            bbox_inches="tight", facecolor=BG)
plt.show()
print("Metrics dashboard saved → fig12_metrics_dashboard.png")

try:
    from google.colab import files
    files.download("fig12_metrics_dashboard.png")
    print("Download triggered")
except Exception:
    pass


Metrics dashboard saved → fig12_metrics_dashboard.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  TRAINING METRICS DASHBOARD
#  Paste this as a new cell in your notebook after training is complete
#  Requires: training logs already printed (data is hardcoded below)
# ══════════════════════════════════════════════════════════════════════

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import numpy as np

# ── Training data (from your logs) ────────────────────────────────────
epochs = np.arange(1, 11)

loss  = [2.0436, 1.6293, 1.4816, 1.3662, 1.2865, 1.2047, 1.1490, 1.0961, 1.0625, 1.0488]
miou  = [0.5208, 0.5369, 0.5094, 0.5380, 0.5114, 0.5071, 0.5210, 0.5262, 0.5316, 0.5343]
mdice = [0.6729, 0.6872, 0.6581, 0.6891, 0.6574, 0.6542, 0.6717, 0.6801, 0.6824, 0.6852]
acc   = [0.7113, 0.7219, 0.6906, 0.7134, 0.7078, 0.6898, 0.7051, 0.7070, 0.7097, 0.7100]
prec  = [0.7097, 0.7242, 0.7179, 0.7236, 0.7699, 0.7756, 0.7861, 0.7732, 0.7721, 0.7607]
rec   = [0.6548, 0.6705, 0.6364, 0.6687, 0.6264, 0.6285, 0.6241, 0.6379, 0.6433, 0.6539]
f1    = [0.6729, 0.6872, 0.6581, 0.6891, 0.6574, 0.6542, 0.6717, 0.6801, 0.6824, 0.6852]
lr_enc= [5.00e-6, 4.88e-6, 4.53e-6, 3.99e-6, 3.31e-6, 2.55e-6, 1.79e-6, 1.11e-6, 5.68e-7, 2.20e-7]

CLASS_NAMES = ['Building', 'Road', 'Water', 'Barren', 'Forest', 'Agriculture', 'Other']

iou_by_class = {
    'Building':    [0.5180, 0.5403, 0.5151, 0.5400, 0.5342, 0.5215, 0.5379, 0.5418, 0.5410, 0.5395],
    'Road':        [0.6174, 0.6600, 0.6667, 0.6677, 0.6757, 0.6657, 0.6416, 0.6226, 0.6570, 0.6721],
    'Water':       [0.5604, 0.5487, 0.5818, 0.5644, 0.5141, 0.5581, 0.5630, 0.5588, 0.5780, 0.5763],
    'Barren':      [0.6909, 0.7093, 0.7108, 0.7191, 0.7217, 0.7209, 0.7205, 0.7100, 0.7179, 0.7132],
    'Forest':      [0.2570, 0.2715, 0.2155, 0.3013, 0.1644, 0.1622, 0.2354, 0.2874, 0.2633, 0.2744],
    'Agriculture': [0.3869, 0.4193, 0.3413, 0.3986, 0.4091, 0.4398, 0.4080, 0.4267, 0.4235, 0.4251],
    'Other':       [0.6153, 0.6090, 0.5348, 0.5750, 0.5610, 0.4816, 0.5405, 0.5363, 0.5403, 0.5394],
}

# Confusion matrix — epoch 4 (best model)
cm_raw = np.array([
    [148250492,  7435926,  2326895,  6993769,  5593082,  4942134,  8466724],
    [  6182914, 29910150,    28441,    41787,    96975,   330670,   140469],
    [  7283602,   108266, 15267212,    12309,   406753,   465372,   260550],
    [  7012451,    35833,    84202, 49352575,   692934,  1237529,  1265865],
    [  9697575,   249308,   119751,   609109,  9682494,   125564,  1762792],
    [ 15361750,    61545,   167521,   137441,   845679, 19952166,  1042829],
    [ 44986441,   172473,   520963,  1154217,  2252864,  5382724, 91205885],
], dtype=np.float64)

# Normalise rows to percentages
row_sums = cm_raw.sum(axis=1, keepdims=True)
cm_pct   = (cm_raw / row_sums * 100).round(1)

# ── Color theme ────────────────────────────────────────────────────────
BG      = "white"
PANEL   = "#f7f7f7"
BORDER  = "#cccccc"
TEXT    = "#1a1a1a"
BLUE    = "#1a73e8"
GREEN   = "#1a7f3c"
RED     = "#c62828"
GOLD    = "#c77700"
PURPLE  = "#5b21b6"
TEAL    = "#0891b2"
GRAY    = "#888888"
LGRID   = "#e8e8e8"

CLASS_COLORS = [BLUE, GREEN, TEAL, GOLD, RED, PURPLE, GRAY]

def style_ax(ax):
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_color(BORDER)
        sp.set_linewidth(0.8)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.title.set_color(TEXT)
    ax.yaxis.grid(True, color=LGRID, linewidth=0.6, zorder=0)
    ax.set_axisbelow(True)

# ══════════════════════════════════════════════════════════════════════
#  FIGURE LAYOUT
# ══════════════════════════════════════════════════════════════════════
fig = plt.figure(figsize=(20, 26), facecolor=BG)
fig.suptitle(
    "DINOv2 Segmentation — Training Metrics Dashboard (10 Epochs)",
    fontsize=16, fontweight="bold", color=TEXT, y=0.99
)

gs = gridspec.GridSpec(
    4, 3,
    figure=fig,
    hspace=0.55, wspace=0.35,
    top=0.96, bottom=0.04
)

# ══ ROW 0 — KPI summary cards ════════════════════════════════════════
kpis = [
    ("Best mIoU",       f"{max(miou):.4f}",    f"Epoch {miou.index(max(miou))+1}",     BLUE),
    ("Best mDice",      f"{max(mdice):.4f}",   f"Epoch {mdice.index(max(mdice))+1}",   GREEN),
    ("Best Accuracy",   f"{max(acc)*100:.2f}%",f"Epoch {acc.index(max(acc))+1}",       GOLD),
    ("Final Loss",      f"{loss[-1]:.4f}",     f"↓ from {loss[0]:.4f}",               RED),
    ("Best F1",         f"{max(f1):.4f}",      f"Epoch {f1.index(max(f1))+1}",         PURPLE),
    ("Best Precision",  f"{max(prec)*100:.2f}%",f"Epoch {prec.index(max(prec))+1}",   TEAL),
]
for col, (title, val, sub, color) in enumerate(kpis[:3]):
    ax = fig.add_subplot(gs[0, col])
    ax.set_facecolor(PANEL)
    for sp in ax.spines.values():
        sp.set_color(color)
        sp.set_linewidth(2.0)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.62, val,   ha="center", va="center", fontsize=36,
            fontweight="bold", color=color, transform=ax.transAxes)
    ax.text(0.5, 0.28, title, ha="center", va="center", fontsize=12,
            color=TEXT, transform=ax.transAxes)
    ax.text(0.5, 0.10, sub,   ha="center", va="center", fontsize=10,
            color=GRAY, transform=ax.transAxes)

# ══ ROW 1 — Loss/mIoU  |  Accuracy/F1  |  LR schedule ═══════════════

# 1A: Loss + mIoU dual axis
ax1 = fig.add_subplot(gs[1, 0])
style_ax(ax1)
ax1.plot(epochs, loss,  color=RED,  lw=2,   marker="o", ms=4, label="Loss")
ax1.set_ylabel("Loss", color=TEXT, fontsize=9)
ax1.set_title("Training loss vs mIoU", fontsize=11, fontweight="bold")
ax1b = ax1.twinx()
ax1b.plot(epochs, miou,  color=BLUE,  lw=2, marker="s", ms=4, ls="--", label="mIoU")
ax1b.plot(epochs, mdice, color=GREEN, lw=2, marker="^", ms=4, ls=":",  label="mDice")
ax1b.set_ylabel("Score", color=TEXT, fontsize=9)
ax1b.tick_params(colors=TEXT, labelsize=9)
ax1b.set_ylim(0.45, 0.75)
# best epoch marker
best_ep = miou.index(max(miou))
ax1b.axvline(epochs[best_ep], color=GOLD, lw=1.2, ls="--", alpha=0.7)
ax1b.text(epochs[best_ep]+0.1, 0.73, f"best ep {epochs[best_ep]}",
          color=GOLD, fontsize=8)
lines1 = [plt.Line2D([0],[0],color=RED,lw=2,marker="o",ms=4),
          plt.Line2D([0],[0],color=BLUE,lw=2,marker="s",ms=4,ls="--"),
          plt.Line2D([0],[0],color=GREEN,lw=2,marker="^",ms=4,ls=":")]
ax1.legend(lines1, ["Loss","mIoU","mDice"], fontsize=8, loc="upper right",
           framealpha=0.9, facecolor=PANEL, edgecolor=BORDER)
ax1.set_xlabel("Epoch", fontsize=9)
ax1.set_xticks(epochs)

# 1B: Accuracy / F1 / Precision / Recall
ax2 = fig.add_subplot(gs[1, 1])
style_ax(ax2)
ax2.plot(epochs, [v*100 for v in acc],  color=PURPLE, lw=2, marker="o", ms=4, label="Accuracy")
ax2.plot(epochs, [v*100 for v in f1],   color=GREEN,  lw=2, marker="s", ms=4, ls="--", label="F1")
ax2.plot(epochs, [v*100 for v in prec], color=TEAL,   lw=2, marker="^", ms=4, ls=":",  label="Precision")
ax2.plot(epochs, [v*100 for v in rec],  color=GOLD,   lw=2, marker="D", ms=4, ls="-.", label="Recall")
ax2.set_ylabel("Score (%)", fontsize=9)
ax2.set_xlabel("Epoch", fontsize=9)
ax2.set_title("Classification metrics", fontsize=11, fontweight="bold")
ax2.set_xticks(epochs)
ax2.set_ylim(58, 82)
ax2.legend(fontsize=8, loc="lower right", framealpha=0.9,
           facecolor=PANEL, edgecolor=BORDER)

# 1C: LR decay
ax3 = fig.add_subplot(gs[1, 2])
style_ax(ax3)
ax3.plot(epochs, [v*1e6 for v in lr_enc], color=RED,  lw=2, marker="o", ms=4)
ax3.fill_between(epochs, [v*1e6 for v in lr_enc], alpha=0.12, color=RED)
ax3.set_ylabel("LR encoder (×10⁻⁶)", fontsize=9)
ax3.set_xlabel("Epoch", fontsize=9)
ax3.set_title("Learning rate schedule (encoder)", fontsize=11, fontweight="bold")
ax3.set_xticks(epochs)

# ══ ROW 2 — Per-class IoU bar  |  Per-class IoU trend ════════════════

# 2A: Per-class IoU best vs final
ax4 = fig.add_subplot(gs[2, 0])
style_ax(ax4)
x = np.arange(len(CLASS_NAMES))
w = 0.35
ep4_iou  = [iou_by_class[c][3] for c in CLASS_NAMES]
ep10_iou = [iou_by_class[c][9] for c in CLASS_NAMES]
bars1 = ax4.bar(x - w/2, ep4_iou,  width=w, color=BLUE,  alpha=0.85,
                edgecolor=BORDER, linewidth=0.5, label="Epoch 4 (best)")
bars2 = ax4.bar(x + w/2, ep10_iou, width=w, color=GRAY,  alpha=0.85,
                edgecolor=BORDER, linewidth=0.5, label="Epoch 10 (final)")
ax4.set_xticks(x)
ax4.set_xticklabels(CLASS_NAMES, fontsize=9, rotation=20, ha="right")
ax4.set_ylabel("IoU", fontsize=9)
ax4.set_ylim(0, 0.85)
ax4.set_title("Per-class IoU: epoch 4 vs epoch 10", fontsize=11, fontweight="bold")
ax4.legend(fontsize=8, framealpha=0.9, facecolor=PANEL, edgecolor=BORDER)
for bar in bars1:
    ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
             f"{bar.get_height():.2f}", ha="center", fontsize=7, color=TEXT)

# 2B: Per-class IoU trend lines
ax5 = fig.add_subplot(gs[2, 1])
style_ax(ax5)
markers = ["o","s","^","D","v","P","X"]
for i, cls in enumerate(CLASS_NAMES):
    ax5.plot(epochs, iou_by_class[cls], color=CLASS_COLORS[i],
             lw=1.8, marker=markers[i], ms=3.5, label=cls)
ax5.set_ylabel("IoU", fontsize=9)
ax5.set_xlabel("Epoch", fontsize=9)
ax5.set_title("Per-class IoU trend across epochs", fontsize=11, fontweight="bold")
ax5.set_xticks(epochs)
ax5.set_ylim(0.1, 0.82)
ax5.legend(fontsize=8, loc="lower right", ncol=2, framealpha=0.9,
           facecolor=PANEL, edgecolor=BORDER)

# 2C: Radar / spider chart — class IoU at best epoch
ax6 = fig.add_subplot(gs[2, 2], polar=True)
ax6.set_facecolor(PANEL)
angles = np.linspace(0, 2*np.pi, len(CLASS_NAMES), endpoint=False).tolist()
angles += angles[:1]
ep4_vals = ep4_iou + [ep4_iou[0]]
ep10_vals= ep10_iou+ [ep10_iou[0]]
ax6.plot(angles, ep4_vals,  color=BLUE,  lw=2,   label="Epoch 4")
ax6.fill(angles, ep4_vals,  color=BLUE,  alpha=0.12)
ax6.plot(angles, ep10_vals, color=GRAY,  lw=1.5, ls="--", label="Epoch 10")
ax6.fill(angles, ep10_vals, color=GRAY,  alpha=0.08)
ax6.set_xticks(angles[:-1])
ax6.set_xticklabels(CLASS_NAMES, fontsize=9, color=TEXT)
ax6.set_ylim(0, 0.85)
ax6.tick_params(colors=TEXT, labelsize=8)
ax6.set_title("Class IoU radar", fontsize=11, fontweight="bold", pad=15, color=TEXT)
ax6.legend(fontsize=8, loc="upper right", bbox_to_anchor=(1.3, 1.1),
           framealpha=0.9, facecolor=BG, edgecolor=BORDER)
ax6.yaxis.grid(True, color=LGRID, linewidth=0.6)
ax6.xaxis.grid(True, color=LGRID, linewidth=0.6)
ax6.spines['polar'].set_color(BORDER)

# ══ ROW 3 — Confusion matrix (full width) ════════════════════════════
ax7 = fig.add_subplot(gs[3, :])
ax7.set_facecolor(BG)
for sp in ax7.spines.values():
    sp.set_visible(False)
ax7.set_xticks([]); ax7.set_yticks([])
ax7.set_title("Confusion matrix — epoch 4 (best model, row-normalised %)",
              fontsize=11, fontweight="bold", color=TEXT, pad=8)

n = len(CLASS_NAMES)
cell_w = 0.10
cell_h = 0.11
x0, y0 = 0.15, 0.02

# header labels
for j, name in enumerate(CLASS_NAMES):
    ax7.text(x0 + (j+0.5)*cell_w, y0 + n*cell_h + 0.04, name,
             ha="center", va="bottom", fontsize=9, color=TEXT, fontweight="500",
             rotation=20)
ax7.text(x0 - 0.11, y0 + n*cell_h/2, "True",
         ha="center", va="center", fontsize=10, color=TEXT,
         rotation=90, fontweight="500")
ax7.text(x0 + n*cell_w/2, y0 - 0.06, "Predicted",
         ha="center", va="center", fontsize=10, color=TEXT, fontweight="500")

for i in range(n):
    row_y = y0 + (n - 1 - i) * cell_h
    ax7.text(x0 - 0.02, row_y + cell_h/2, CLASS_NAMES[i],
             ha="right", va="center", fontsize=9, color=TEXT, fontweight="500")
    for j in range(n):
        val = cm_pct[i, j]
        if i == j:
            intensity = min(0.9, val/100 * 1.3 + 0.1)
            bg = (*[v/255 for v in (26, 115, 232)], intensity)
            fc = "#0c447c" if intensity > 0.4 else TEXT
        else:
            intensity = min(0.65, val/100 * 10)
            bg = (*[v/255 for v in (198, 40, 40)], intensity)
            fc = "#7f1d1d" if intensity > 0.25 else TEXT

        rect = plt.Rectangle(
            (x0 + j*cell_w, row_y), cell_w, cell_h,
            transform=ax7.transAxes,
            facecolor=bg, edgecolor="white", linewidth=0.8,
            clip_on=False
        )
        ax7.add_patch(rect)
        ax7.text(x0 + (j+0.5)*cell_w, row_y + cell_h/2, f"{val:.1f}%",
                 ha="center", va="center", fontsize=8.5,
                 fontweight="500" if i==j else "400",
                 color=fc, transform=ax7.transAxes)

# ── Save & show ────────────────────────────────────────────────────────
plt.savefig("training_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor=BG)
plt.show()
print("Dashboard saved → training_dashboard.png")

try:
    from google.colab import files
    files.download("training_dashboard.png")
    print("Download triggered")
except Exception:
    pass


Dashboard saved → training_dashboard.png


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered


In [ ]:
# ================================================================
# MULTI-IMAGE LOVE-DA RURAL TEST IMAGE UPLOAD
# ================================================================

from google.colab import files
import os

print("Select 10–20 different LoveDA Rural test images.")
print("Location on your PC:")
print(r"D:\new_MPS\remoteSensingWebApp\LoveDA Satellite Images Semantic Segmentation\Test\Test\Rural\images_png")

uploaded = files.upload()

test_image_paths = sorted([
    os.path.join("/content", name)
    for name in uploaded.keys()
    if name.lower().endswith((".png", ".jpg", ".jpeg"))
])

print("\nUploaded images:", len(test_image_paths))

for path in test_image_paths:
    print(os.path.basename(path))

if len(test_image_paths) < 10:
    print("\n⚠️ Please upload at least 10 different images.")
elif len(test_image_paths) > 20:
    print("\n⚠️ You uploaded more than 20 images. We can use the first 20.")
    test_image_paths = test_image_paths[:20]
else:
    print("\n✅ Ready for multi-image evaluation.")

Select 10–20 different LoveDA Rural test images.
Location on your PC:
D:\new_MPS\remoteSensingWebApp\LoveDA Satellite Images Semantic Segmentation\Test\Test\Rural\images_png


Saving 4211.png to 4211 (1).png
Saving 4212.png to 4212 (1).png
Saving 4213.png to 4213 (1).png
Saving 4214.png to 4214 (1).png
Saving 4215.png to 4215.png
Saving 4216.png to 4216 (1).png
Saving 4217.png to 4217 (1).png
Saving 4218.png to 4218 (1).png
Saving 4219.png to 4219 (1).png
Saving 4220.png to 4220.png

Uploaded images: 10
4211 (1).png
4212 (1).png
4213 (1).png
4214 (1).png
4215.png
4216 (1).png
4217 (1).png
4218 (1).png
4219 (1).png
4220.png

✅ Ready for multi-image evaluation.


In [ ]:
# ================================================================
# MULTI-IMAGE PLANNING EVALUATION
# Uses the existing trained model + InfrastructurePlanner
# NO RETRAINING
# ================================================================

import os
import io
import contextlib
import numpy as np
import pandas as pd

batch_results = []

print("Running planning evaluation on", len(test_image_paths), "images...\n")

for idx, image_path in enumerate(test_image_paths, 1):

    image_name = os.path.basename(image_path)

    print(f"[{idx}/{len(test_image_paths)}] Processing {image_name} ...")

    try:
        # --------------------------------------------------------
        # 1. Predict segmentation using existing trained model
        # --------------------------------------------------------
        with contextlib.redirect_stdout(io.StringIO()):
            original_img, pred_mask = predict_single(image_path)

        # --------------------------------------------------------
        # 2. Run existing infrastructure planner
        # --------------------------------------------------------
        with contextlib.redirect_stdout(io.StringIO()):
            planner = InfrastructurePlanner(
                original_img,
                pred_mask,
                min_building_area=20,
                road_kernel_size=3,
                connectivity_radius=5
            )

            results = planner.run()

        # --------------------------------------------------------
        # 3. Get planning metrics
        # --------------------------------------------------------
        m = results.metrics

        row = {
            "image": image_name,

            "total_buildings":
                m.get("total_buildings", 0),

            "connected_before":
                m.get("connected_buildings_before", 0),

            "isolated_before":
                m.get("isolated_buildings_before", 0),

            "connectivity_before_pct":
                m.get("connectivity_before_pct", 0),

            "connected_after":
                m.get("connected_buildings_after", 0),

            "connectivity_after_pct":
                m.get("connectivity_after_pct", 0),

            "connectivity_gain_pct":
                m.get("connectivity_after_pct", 0)
                - m.get("connectivity_before_pct", 0),

            "proposed_connections":
                m.get("proposed_new_road_connections", 0),

            "total_proposed_road_length_px":
                m.get("total_proposed_road_length_px", 0),

            "average_proposed_road_length_px":
                m.get("average_proposed_road_length_px", 0),

            "avg_connection_distance_px":
                m.get("avg_connection_distance_px", 0),

            "average_path_efficiency":
                m.get("average_path_efficiency", 0),
        }

        batch_results.append(row)

        print(
            f"   Buildings: {row['total_buildings']} | "
            f"Before: {row['connectivity_before_pct']:.1f}% | "
            f"After: {row['connectivity_after_pct']:.1f}% | "
            f"Proposed: {row['proposed_connections']} | "
            f"Efficiency: {row['average_path_efficiency']:.3f}"
        )

    except Exception as e:

        print(f"   ❌ Error: {e}")

        batch_results.append({
            "image": image_name,
            "error": str(e)
        })

    # Free GPU memory after each image
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# ================================================================
# CREATE RESULTS TABLE
# ================================================================

planning_df = pd.DataFrame(batch_results)

print("\n" + "=" * 80)
print("MULTI-IMAGE PLANNING EVALUATION RESULTS")
print("=" * 80)

display(planning_df)

Running planning evaluation on 10 images...

[1/10] Processing 4211 (1).png ...
   Buildings: 9 | Before: 100.0% | After: 100.0% | Proposed: 0 | Efficiency: 0.000
[2/10] Processing 4212 (1).png ...
   Buildings: 1 | Before: 0.0% | After: 100.0% | Proposed: 1 | Efficiency: 0.931
[3/10] Processing 4213 (1).png ...
   Buildings: 2 | Before: 50.0% | After: 100.0% | Proposed: 1 | Efficiency: 0.771
[4/10] Processing 4214 (1).png ...
   Buildings: 0 | Before: 0.0% | After: 0.0% | Proposed: 0 | Efficiency: 0.000
[5/10] Processing 4215.png ...
   Buildings: 18 | Before: 55.6% | After: 100.0% | Proposed: 8 | Efficiency: 0.904
[6/10] Processing 4216 (1).png ...
   Buildings: 14 | Before: 28.6% | After: 100.0% | Proposed: 10 | Efficiency: 0.859
[7/10] Processing 4217 (1).png ...
   Buildings: 0 | Before: 0.0% | After: 0.0% | Proposed: 0 | Efficiency: 0.000
[8/10] Processing 4218 (1).png ...
   Buildings: 4 | Before: 50.0% | After: 100.0% | Proposed: 2 | Efficiency: 0.943
[9/10] Processing 4219 (1)

,image,total_buildings,connected_before,isolated_before,connectivity_before_pct,connected_after,connectivity_after_pct,connectivity_gain_pct,proposed_connections,total_proposed_road_length_px,average_proposed_road_length_px,avg_connection_distance_px,average_path_efficiency
0,4211 (1).png,9,9,0,100.00,9,100.0,0.00,0,0.00,0.00,0.00,0.0000
1,4212 (1).png,1,0,1,0.00,1,100.0,100.00,1,37.14,37.14,34.60,0.9315
2,4213 (1).png,2,1,1,50.00,2,100.0,50.00,1,201.23,201.23,155.17,0.7711
3,4214 (1).png,0,0,0,0.00,0,0.0,0.00,0,0.00,0.00,0.00,0.0000
4,4215.png,18,10,8,55.56,18,100.0,44.44,8,906.68,113.34,102.72,0.9036
5,4216 (1).png,14,4,10,28.57,14,100.0,71.43,10,2029.94,202.99,170.78,0.8594
6,4217 (1).png,0,0,0,0.00,0,0.0,0.00,0,0.00,0.00,0.00,0.0000
7,4218 (1).png,4,2,2,50.00,4,100.0,50.00,2,296.02,148.01,139.24,0.9426
8,4219 (1).png,16,0,16,0.00,0,0.0,0.00,0,0.00,0.00,0.00,0.0000
9,4220.png,1,1,0,100.00,1,100.0,0.00,0,0.00,0.00,0.00,0.0000


In [ ]:
# ================================================================
# DEBUG: FIND EXACT LINE CAUSING "list index out of range"
# ================================================================

import traceback
import os

test_path = test_image_paths[0]

print("Testing:", os.path.basename(test_path))
print("=" * 70)

try:
    print("\n1. Running prediction...")
    original_img, pred_mask = predict_single(test_path)
    print("   ✅ Prediction successful")
    print("   Image shape:", original_img.shape)
    print("   Prediction shape:", pred_mask.shape)
    print("   Classes found:", np.unique(pred_mask))

    print("\n2. Creating planner...")
    planner = InfrastructurePlanner(
        original_img,
        pred_mask,
        min_building_area=20,
        road_kernel_size=3,
        connectivity_radius=5
    )
    print("   ✅ Planner created")

    print("\n3. Running planner...")
    results = planner.run()
    print("   ✅ Planner completed")

except Exception as e:
    print("\n❌ ERROR:")
    print(type(e).__name__, ":", str(e))
    print("\nFULL TRACEBACK:")
    traceback.print_exc()

Testing: 4211 (1).png

1. Running prediction...

Detected Classes:
  Building       : 43339 px  (13.8%)
  Road           : 53365 px  (17.0%)
  Water          : 10564 px  (3.4%)
  Barren         : 9960 px  (3.2%)
  Agriculture    : 12163 px  (3.9%)
  Other          : 184209 px  (58.7%)
  Building       : 43339 px  (13.8%)
  Road           : 53365 px  (17.0%)
  Water          : 10564 px  (3.4%)
  Barren         : 9960 px  (3.2%)
  Agriculture    : 12163 px  (3.9%)
  Other          : 184209 px  (58.7%)
  Building       : 43339 px  (13.8%)
  Road           : 53365 px  (17.0%)
  Water          : 10564 px  (3.4%)
  Barren         : 9960 px  (3.2%)
  Agriculture    : 12163 px  (3.9%)
  Other          : 184209 px  (58.7%)
  Building       : 43339 px  (13.8%)
  Road           : 53365 px  (17.0%)
  Water          : 10564 px  (3.4%)
  Barren         : 9960 px  (3.2%)
  Agriculture    : 12163 px  (3.9%)
  Other          : 184209 px  (58.7%)
  Building       : 43339 px  (13.8%)
  Road           : 5

In [ ]:
# ================================================================
# MULTI-IMAGE PLANNING SUMMARY
# Mean ± Standard Deviation
# ================================================================

import numpy as np
import pandas as pd

summary_metrics = [
    "total_buildings",
    "connectivity_before_pct",
    "connectivity_after_pct",
    "connectivity_gain_pct",
    "proposed_connections",
    "total_proposed_road_length_px",
    "average_proposed_road_length_px",
    "avg_connection_distance_px",
    "average_path_efficiency"
]

summary_rows = []

for metric in summary_metrics:

    values = pd.to_numeric(
        planning_df[metric],
        errors="coerce"
    ).dropna()

    summary_rows.append({
        "Metric": metric,
        "Mean": values.mean(),
        "Std": values.std(ddof=1),
        "Mean ± Std": f"{values.mean():.2f} ± {values.std(ddof=1):.2f}"
    })

summary_df = pd.DataFrame(summary_rows)

print("=" * 80)
print("MULTI-IMAGE PLANNING SUMMARY")
print("=" * 80)

display(summary_df)

MULTI-IMAGE PLANNING SUMMARY


,Metric,Mean,Std,Mean ± Std
0,total_buildings,6.50000,7.121954,6.50 ± 7.12
1,connectivity_before_pct,38.41300,39.647857,38.41 ± 39.65
2,connectivity_after_pct,70.00000,48.304589,70.00 ± 48.30
3,connectivity_gain_pct,31.58700,36.666322,31.59 ± 36.67
4,proposed_connections,2.20000,3.675746,2.20 ± 3.68
5,total_proposed_road_length_px,347.10100,655.526482,347.10 ± 655.53
6,average_proposed_road_length_px,70.27100,87.210227,70.27 ± 87.21
7,avg_connection_distance_px,60.25100,73.089020,60.25 ± 73.09
8,average_path_efficiency,0.44082,0.466979,0.44 ± 0.47


In [ ]:
# ================================================================
# CORRECTED MULTI-IMAGE PLANNING SUMMARY
# Exclude N/A values for path-based metrics
# ================================================================

df = planning_df.copy()

# Path-based metrics are meaningful only when at least one
# proposed connection exists.
valid_path = df["proposed_connections"] > 0

print("=" * 80)
print("CORRECTED MULTI-IMAGE PLANNING SUMMARY")
print("=" * 80)

# ------------------------------------------------
# 1. Image-level statistics
# ------------------------------------------------
print("\nIMAGE-LEVEL RESULTS")
print("-" * 80)

print(f"Number of images evaluated       : {len(df)}")
print(f"Images with proposed paths      : {valid_path.sum()}")
print(f"Images without proposed paths   : {(~valid_path).sum()}")

# ------------------------------------------------
# 2. Building-level pooled connectivity
# ------------------------------------------------
total_buildings = df["total_buildings"].sum()
total_connected_before = df["connected_before"].sum()
total_connected_after = df["connected_after"].sum()
total_isolated = df["isolated_before"].sum()
total_proposed = df["proposed_connections"].sum()

pooled_before = (
    total_connected_before / total_buildings * 100
    if total_buildings > 0 else 0
)

pooled_after = (
    total_connected_after / total_buildings * 100
    if total_buildings > 0 else 0
)

pooled_gain = pooled_after - pooled_before

print("\nBUILDING-LEVEL CONNECTIVITY")
print("-" * 80)

print(f"Total buildings                  : {total_buildings}")
print(f"Connected before planning        : {total_connected_before}")
print(f"Isolated before planning         : {total_isolated}")
print(f"Connected after planning         : {total_connected_after}")
print(f"Pooled connectivity before (%)   : {pooled_before:.2f}")
print(f"Pooled connectivity after (%)    : {pooled_after:.2f}")
print(f"Connectivity improvement (pp)    : {pooled_gain:.2f}")

# ------------------------------------------------
# 3. Proposed road statistics
# ------------------------------------------------
total_road_length = df["total_proposed_road_length_px"].sum()

print("\nPROPOSED ROAD STATISTICS")
print("-" * 80)

print(f"Total proposed connections       : {total_proposed}")
print(f"Total proposed road length (px)  : {total_road_length:.2f}")

if total_proposed > 0:
    pooled_avg_road_length = total_road_length / total_proposed
else:
    pooled_avg_road_length = 0

print(f"Average road length / connection : {pooled_avg_road_length:.2f} px")

# ------------------------------------------------
# 4. Path-based metrics
# ------------------------------------------------
efficiency = pd.to_numeric(
    df.loc[valid_path, "average_path_efficiency"],
    errors="coerce"
).dropna()

connection_distance = pd.to_numeric(
    df.loc[valid_path, "avg_connection_distance_px"],
    errors="coerce"
).dropna()

avg_road_length = pd.to_numeric(
    df.loc[valid_path, "average_proposed_road_length_px"],
    errors="coerce"
).dropna()

print("\nPATH-BASED METRICS")
print("-" * 80)

print(
    f"Path efficiency                 : "
    f"{efficiency.mean():.3f} ± {efficiency.std(ddof=1):.3f}"
)

print(
    f"Connection distance (px)        : "
    f"{connection_distance.mean():.2f} ± "
    f"{connection_distance.std(ddof=1):.2f}"
)

print(
    f"Average road length (px)        : "
    f"{avg_road_length.mean():.2f} ± "
    f"{avg_road_length.std(ddof=1):.2f}"
)

print("=" * 80)

CORRECTED MULTI-IMAGE PLANNING SUMMARY

IMAGE-LEVEL RESULTS
--------------------------------------------------------------------------------
Number of images evaluated       : 10
Images with proposed paths      : 5
Images without proposed paths   : 5

BUILDING-LEVEL CONNECTIVITY
--------------------------------------------------------------------------------
Total buildings                  : 65
Connected before planning        : 27
Isolated before planning         : 38
Connected after planning         : 49
Pooled connectivity before (%)   : 41.54
Pooled connectivity after (%)    : 75.38
Connectivity improvement (pp)    : 33.85

PROPOSED ROAD STATISTICS
--------------------------------------------------------------------------------
Total proposed connections       : 22
Total proposed road length (px)  : 3471.01
Average road length / connection : 157.77 px

PATH-BASED METRICS
--------------------------------------------------------------------------------
Path efficiency               

In [ ]:
import numpy as np
import pandas as pd

# Epoch-4 validation confusion matrix
# Rows = Ground Truth
# Columns = Predicted
cm = np.array([
    [148250492, 7435926, 2326895, 6993769, 5593082, 4942134, 8466724],
    [6182914, 29910150, 28441, 41787, 96975, 330670, 140469],
    [7283602, 108266, 15267212, 12309, 406753, 465372, 260550],
    [7012451, 35833, 84202, 49352575, 692934, 1237529, 1265865],
    [9697575, 249308, 119751, 609109, 9682494, 125564, 1762792],
    [15361750, 61545, 167521, 137441, 845679, 19952166, 1042829],
    [44986441, 172473, 520963, 1154217, 2252864, 5382724, 91205885]
], dtype=np.int64)

classes = [
    "Building",
    "Road",
    "Water",
    "Barren",
    "Forest",
    "Agriculture",
    "Other"
]

results = []

for i, cls in enumerate(classes):

    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0 else 0
    )
    iou = TP / (TP + FP + FN) if (TP + FP + FN) > 0 else 0

    results.append([
        cls,
        iou * 100,
        precision * 100,
        recall * 100,
        f1 * 100
    ])

df = pd.DataFrame(
    results,
    columns=["Class", "IoU (%)", "Precision (%)", "Recall (%)", "F1 (%)"]
)

print(df.to_string(
    index=False,
    formatters={
        "IoU (%)": "{:.2f}".format,
        "Precision (%)": "{:.2f}".format,
        "Recall (%)": "{:.2f}".format,
        "F1 (%)": "{:.2f}".format
    }
))

      Class IoU (%) Precision (%) Recall (%) F1 (%)
   Building   54.00         62.09      80.57  70.13
       Road   66.77         78.77      81.43  80.08
      Water   56.44         82.46      64.14  72.15
     Barren   71.91         84.65      82.69  83.66
     Forest   30.13         49.47      43.52  46.31
Agriculture   39.86         61.51      53.11  57.00
      Other   57.50         87.58      62.61  73.02
